In [ ]:
import sys
sys.path.append("..")

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

from pipeline.ingesta import cargar_indices_desde_bd

# Definición de rutas del proyecto
RUTA_CLASIFICADORES = "../clasificadores/"
RUTA_BASE_DATOS = "../data/pipeline.gpkg"

print("¡Entorno configurado correctamente!")

In [ ]:
# Lista los nombres de tus archivos .pkl exactos en la carpeta /clasificadores
# Ejemplo: ['rf_evi.pkl', 'rf_lswi.pkl', 'rf_combinado.pkl']
modelos_archivos = {
    "modelo_1": "random_forest_maiz_NDVI_45pasos_v1.pkl",  # Cambia por tus nombres reales
    "modelo_2": "rf_maiz_comayagua_optimizado_NDVI_9pasos.pkl",
    "modelo_3": "rf_maiz_comayagua_evi_ndre.pkl"
}

modelos = {}
for nombre, archivo in modelos_archivos.items():
    ruta_modelo = os.path.join(RUTA_CLASIFICADORES, archivo)
    if os.path.exists(ruta_modelo):
        modelos[nombre] = joblib.load(ruta_modelo)
        print(f"-> {nombre} cargado con éxito desde {archivo}")
    else:
        print(f"⚠️ Alerta: No se encontró el archivo en {ruta_modelo}")

In [ ]:
# 1. Cargar los índices crudos desde el GeoPackage (puedes filtrar por fechas o ids si lo deseas)
# Si dejas los parámetros vacíos traerá todo el histórico disponible
dict_indices = cargar_indices_desde_bd(
    fecha_inicio=None, 
    fecha_fin=None, 
    ids_parcelas=None
)

df_evi = dict_indices["EVI"]
df_lswi = dict_indices["LSWI"]

print(f"Datos originales de BD (Fechas x Columnas de Parcelas):")
print(f" - EVI shape: {df_evi.shape} | LSWI shape: {df_lswi.shape}")

# 2. Transponer para tener Formato: (Parcelas x Pasos de Tiempo)
# Pasamos de columnas 'id_123' a un índice limpio con solo el número entero
X_evi = df_evi.T
X_evi.index = X_evi.index.str.replace("id_", "").astype(int)

X_lswi = df_lswi.T
X_lswi.index = X_lswi.index.str.replace("id_", "").astype(int)

# Guardamos la lista de IDs de parcelas para mapear los resultados finales
ids_parcelas = X_evi.index.tolist()

# 3. Validar dimensiones de entrada según tu entrenamiento (ej. 45 observaciones o una ventana)
# NOTA: Ajusta el recorte de columnas/ventanas temporales si tus modelos usan subconjuntos específicos
print(f"\nMatrices transformadas listas para inferencia:")
print(f" - X_evi (Parcelas, Características): {X_evi.shape}")

In [ ]:
# Crear un DataFrame base para consolidar las predicciones por parcela
df_resultados = pd.DataFrame(index=ids_parcelas)
df_resultados.index.name = "id_parcela"

# Ejecutar inferencia para cada modelo cargado
# NOTA: Asegúrate de pasar la matriz correcta (X_evi, X_lswi o concatenadas) según cada modelo
for nombre_modelo, model_rf in modelos.items():
    try:
        # Supongamos por defecto que tus modelos principales se entrenaron con el perfil de EVI
        # Si un modelo usa combinación (ej. EVI + LSWI), concatena horizontalmente: np.hstack((X_evi, X_lswi))
        X_input = X_evi.values 
        
        # Validación de features esperadas vs entregadas
        n_features_modelo = model_rf.n_features_in_
        if X_input.shape[1] != n_features_modelo:
            print(f"⚙️ Ajustando dimensiones para {nombre_modelo}: requiere {n_features_modelo} pasos.")
            # Ajustar eje temporal truncando o seleccionando la ventana del entrenamiento (ej. pasos 4 al 12)
            X_input = X_input[:, :n_features_modelo] 
        
        # Predicción e Inferencia
        preds = model_rf.predict(X_input)
        probabilidades = model_rf.predict_proba(X_input)[:, 1] # Probabilidad de ser Maíz (Clase 1)
        
        # Guardar en el DataFrame de resultados
        df_resultados[f"pred_{nombre_modelo}"] = preds
        df_resultados[f"prob_{nombre_modelo}"] = probabilidades
        
    except Exception as e:
        print(f"❌ Error al ejecutar inferencia con {nombre_modelo}: {e}")

print("Inferencia completada. Muestra de los resultados:")
print(df_resultados.head())

In [ ]:
# 1. Cargar las geometrías de las parcelas desde el Geopackage
# Reemplaza 'capa_parcelas_test' por el nombre real de la capa de tus polígonos
layer_name_parcelas = "parcelas_comayagua" 

gdf_parcelas = gpd.read_file(RUTA_BASE_DATOS, layer=layer_name_parcelas)

# Asegurar que el id de unión sea numérico para hacer el merge correcto
gdf_parcelas["id_parcela"] = gdf_parcelas["id_parcela"].astype(int)

# 2. Unir polígonos con las predicciones generadas
gdf_final = gdf_parcelas.merge(df_resultados, on="id_parcela", how="inner")

# 3. Guardar el nuevo set de datos validado en el pipeline.gpkg con pyogrio
capa_salida = "validacion_predicciones_rf"
modo_pyogrio = "w" # 'w' para sobreescribir/crear, 'a' para añadir

gdf_final.to_file(
    RUTA_BASE_DATOS,
    layer=capa_salida,
    driver="GPKG",
    mode=modo_pyogrio,
)

print(f"🎉 ¡Validación exitosa! Resultados guardados en '{RUTA_BASE_DATOS}' bajo la capa '{capa_salida}'.")
print(f"Total de parcelas evaluadas e indexadas: {len(gdf_final)}")